# AlzDetect — Can we honestly beat 61%? (CV fine-tuning)

The frozen-feature baseline tops out at ~61% acc / ~0.56 macro-F1 under clean 5-fold CV. This notebook tests the two **legitimate, leakage-free** levers to improve it:

1. **Fine-tune the MobileNetV2 backbone** (unfreeze the top block) per fold.
2. **Train-fold-only augmentation** (applied AFTER the split — no leakage).

Protocol: same stratified 5-fold CV, report **mean ± std**. Test fold is never augmented and never seen in training. Compare against the frozen-feature SMOTE result (acc 0.612 / macro-F1 0.558).

**Setup:** Add Input `alzheimermridataset` (legendahmed); Internet ON; GPU P100. (~5-10 min/fold.)

In [ ]:
!pip install -q imbalanced-learn
import os, glob, re
import numpy as np, cv2
cands = glob.glob('/kaggle/input/**/all image', recursive=True)
assert cands, 'Add the legendahmed/alzheimermridataset input.'
DATA_DIR = cands[0]; print('DATA_DIR =', DATA_DIR)

IMG_SIZE, N_SPLITS, SEED = 224, 5, 42
np.random.seed(SEED)
PREFIX={'mildDem':'Mild Dementia','moderateDem':'Moderate Dementia','verymildDem':'Very mild Dementia','nonDem':'Non Demented'}
CLASS_ORDER=['Mild Dementia','Moderate Dementia','Very mild Dementia','Non Demented']
cmap={c:i for i,c in enumerate(CLASS_ORDER)}
imgs,labels=[],[]
for fn in sorted(os.listdir(DATA_DIR)):
    m=re.match(r'^[a-zA-Z]+',fn)
    if m and PREFIX.get(m.group()):
        im=cv2.imread(os.path.join(DATA_DIR,fn))
        if im is not None:
            imgs.append(cv2.resize(im,(IMG_SIZE,IMG_SIZE))); labels.append(cmap[PREFIX[m.group()]])
X=np.array(imgs,dtype=np.uint8); y=np.array(labels); del imgs
print('Loaded', X.shape, {CLASS_ORDER[i]:int((y==i).sum()) for i in range(4)})

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

FINE_TUNE_LAYERS = 30   # how many top MobileNetV2 layers to unfreeze

def build_model():
    inp=Input((IMG_SIZE,IMG_SIZE,3))
    base=MobileNetV2(weights='imagenet',include_top=False,input_tensor=inp)
    base.trainable=False
    x=GlobalAveragePooling2D()(base.output); x=Dropout(0.3)(x)
    out=Dense(4,activation='softmax')(x)
    return Model(inp,out), base

# Train-fold-only augmentation (structure-preserving). raw 0-255 (no rescale).
aug = ImageDataGenerator(rotation_range=10, width_shift_range=0.05,
                         height_shift_range=0.05, zoom_range=0.1, horizontal_flip=True)

def run_fold(tr, te, seed):
    tf.keras.utils.set_random_seed(seed)
    Xtr, ytr = X[tr].astype('float32'), y[tr]
    Xte, yte = X[te].astype('float32'), y[te]
    # internal val split for early stopping (no test leakage)
    Xtr2,Xv,ytr2,yv = train_test_split(Xtr,ytr,test_size=0.1,stratify=ytr,random_state=seed)
    cw=compute_class_weight('balanced',classes=np.arange(4),y=ytr2); cw={i:cw[i] for i in range(4)}
    model, base = build_model()
    flow = aug.flow(Xtr2, to_categorical(ytr2,4), batch_size=32, seed=seed)
    es=[EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)]
    # Phase 1: frozen base, train head
    model.compile(optimizer=Adam(1e-3),loss='categorical_crossentropy',metrics=['accuracy'])
    model.fit(flow,validation_data=(Xv,to_categorical(yv,4)),epochs=8,class_weight=cw,callbacks=es,verbose=0)
    # Phase 2: unfreeze top block, fine-tune at low LR
    for l in base.layers[-FINE_TUNE_LAYERS:]: l.trainable=True
    model.compile(optimizer=Adam(1e-5),loss='categorical_crossentropy',metrics=['accuracy'])
    model.fit(flow,validation_data=(Xv,to_categorical(yv,4)),epochs=25,class_weight=cw,callbacks=es,verbose=0)
    yp=np.argmax(model.predict(Xte,verbose=0),axis=1)
    return accuracy_score(yte,yp), f1_score(yte,yp,average='macro')

accs,f1s=[],[]
skf=StratifiedKFold(n_splits=N_SPLITS,shuffle=True,random_state=SEED)
for fold,(tr,te) in enumerate(skf.split(X,y)):
    a,f=run_fold(tr,te,SEED+fold); accs.append(a); f1s.append(f)
    print(f'Fold {fold+1}: acc={a:.4f}  macroF1={f:.4f}')
accs=np.array(accs); f1s=np.array(f1s)
print(f'\nFine-tuned + train-only aug (5-fold CV):')
print(f'  Accuracy  = {accs.mean():.3f} ± {accs.std():.3f}')
print(f'  Macro-F1  = {f1s.mean():.3f} ± {f1s.std():.3f}')
print(f'\nFrozen-feature SMOTE baseline was: acc 0.612 / macro-F1 0.558')